In [1]:
pip install -q transformers datasets peft trl accelerate bitsandbytes mlflow scikit-learn pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 120.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)

messages = [{"role": "user", "content": "In one sentence, what is a 401(k) plan?"}]
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=60)
print(tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

A 401(k) plan is a retirement savings plan offered by employers to employees in the United States.


In [3]:
from pydantic import BaseModel, Field
from typing import Literal

INTENT_DESCRIPTIONS = {
    "TRANSFER_REQUEST": "explicitly asks to move or roll over money out of the plan",
    "FEE_QUESTION": "asks about fees, costs, or expense ratios",
    "COMPARISON_REQUEST": "wants options compared - keep in plan vs roll over",
    "TAX_ADVICE": "asks about tax consequences or tax strategy",
    "PLAN_QUESTION": "asks about plan rules - vesting, eligibility, employer match",
    "ACCOUNT_SUPPORT": "login, balance, statement, or other account help",
    "COMPLAINT": "expresses dissatisfaction or frustration with the service",
    "OTHER": "anything else",
}

INTENT_LABELS = list(INTENT_DESCRIPTIONS.keys())

class IntentPrediction(BaseModel):
    intent: Literal[tuple(INTENT_LABELS)]
    confidence: float = Field(ge=0.0, le=1.0)
    reason: str = Field(max_length=140)

for label, description in INTENT_DESCRIPTIONS.items():
    print(f"{label}: {description}")

TRANSFER_REQUEST: explicitly asks to move or roll over money out of the plan
FEE_QUESTION: asks about fees, costs, or expense ratios
COMPARISON_REQUEST: wants options compared - keep in plan vs roll over
TAX_ADVICE: asks about tax consequences or tax strategy
PLAN_QUESTION: asks about plan rules - vesting, eligibility, employer match
ACCOUNT_SUPPORT: login, balance, statement, or other account help
COMPLAINT: expresses dissatisfaction or frustration with the service
OTHER: anything else


In [4]:
import json, random
from collections import Counter

random.seed(42)

SLOTS = {
    "provider": ["Fidelity", "Voya", "Empower", "Vanguard", "my bank"],
    "company": ["Harbor Manufacturing", "Lakeshore Services", "my employer", "my old employer"],
    "amount": ["$4,000", "$12,500", "$23,000", "$850", "$67,000", "$150,000"],
    "year": ["2024", "2025", "2026"],
    "age": ["59 and a half", "62", "65"],
}

TEMPLATES = {
    "TRANSFER_REQUEST": [
        "I want to roll my 401k over to an IRA",
        "How do I move my money out of the plan",
        "Transfer my balance to my IRA at {provider} please",
        "I'd like to cash out my 401k",
        "Move everything to a rollover IRA",
        "Can you start the rollover process for my account",
        "I left {company} and want my {amount} moved out",
        "Just transfer it all, I don't want to compare anything",
        "Please send my vested balance to my bank",
        "I need to withdraw my 401k money",
        "Get my money out of this plan",
        "Roll my {amount} into my existing IRA",
    ],
    "FEE_QUESTION": [
        "What are the fees in this plan",
        "How much am I paying in expense ratios",
        "What does the plan charge me every year",
        "Are there fees for rolling over",
        "What's the expense ratio on the target date fund",
        "How much do I pay in administrative fees",
        "What are the fund fees in my 401k",
        "Is there a charge to move my money",
        "What am I being charged for the managed account service",
        "How do the fees here compare to an IRA",
        "What percentage goes to fees each year",
        "Are there any hidden costs in the plan",
    ],
    "COMPARISON_REQUEST": [
        "Should I keep my money in the plan or roll it over",
        "Can you compare my options for me",
        "What are the pros and cons of leaving my 401k where it is",
        "Help me decide between the plan and an IRA",
        "Is it better to stay in the plan or move to {provider}",
        "Walk me through keep vs rollover",
        "Compare the costs of staying versus rolling over",
        "I'm retiring in {year} - should the money stay or go",
        "Lay out my options side by side",
        "Does it make sense to consolidate everything into an IRA",
        "What do I gain by keeping it in the plan",
        "Which is smarter for {amount}, stay or roll",
    ],
    "TAX_ADVICE": [
        "Will I owe taxes if I roll over to a Roth",
        "What are the tax consequences of cashing out",
        "How much tax will I pay on a withdrawal at {age}",
        "Can I avoid the 10 percent penalty",
        "Is a direct rollover taxable",
        "Should I do a Roth conversion this year to save on taxes",
        "What tax bracket will my distribution fall into",
        "How do I minimize taxes on my 401k withdrawal",
        "Do I pay state tax on a rollover",
        "What's the penalty for early withdrawal",
        "Will cashing out count as income this year",
        "Tell me the smartest tax move for my balance",
    ],
    "PLAN_QUESTION": [
        "What is the vesting schedule",
        "When am I eligible to contribute",
        "What is the employer match",
        "How does the match vest at {company}",
        "When can I enroll in the plan",
        "Am I fully vested yet",
        "What's the maximum I can contribute in {year}",
        "Does the plan allow catch-up contributions",
        "How often is the match deposited",
        "Can I take a loan from my 401k",
        "What happens to my match if I leave before three years",
        "When do employer contributions become mine",
    ],
    "ACCOUNT_SUPPORT": [
        "I can't log into my account",
        "How do I reset my password",
        "Where do I find my statement",
        "My balance looks wrong",
        "The app keeps crashing when I sign in",
        "How do I update my address",
        "I never got my confirmation email",
        "Can you mail me a paper statement",
        "How do I change my beneficiaries online",
        "The website says my account is locked",
        "Where's my 1099 form",
        "How do I set up direct deposit for distributions",
    ],
    "COMPLAINT": [
        "This is ridiculous, nobody answers the phone",
        "I've been waiting three weeks for my distribution",
        "Your website lost my paperwork twice",
        "I'm furious about the fees you charged me",
        "This is the worst service I've ever dealt with",
        "Why is it so hard to get my own money",
        "I want to file a formal complaint",
        "Your rep gave me wrong information last week",
        "Unacceptable - my rollover has been pending for a month",
        "I'm sick of getting transferred between departments",
        "This company doesn't care about participants",
        "Extremely disappointed with how this was handled",
    ],
    "OTHER": [
        "What time does the call center open",
        "Thanks for your help yesterday",
        "Can I get a copy of the plan highlights brochure",
        "Do you have an office in Hartford",
        "What's the weather like there",
        "I was just checking how this chat works",
        "Happy holidays to the team",
        "Is this a real person",
        "What languages do you support",
        "How do I unsubscribe from marketing emails",
        "Just saying hi",
        "Do you have a mobile app",
    ],
}

GREETINGS = ["Hi, ", "Hello, ", "Hey, ", "Good morning, "]

def fill(template):
    out = template
    for key, values in SLOTS.items():
        token = "{" + key + "}"
        while token in out:
            out = out.replace(token, random.choice(values), 1)
    return out

def mutate(text):
    if random.random() < 0.25:
        text = random.choice(GREETINGS) + text[0].lower() + text[1:]
    if random.random() < 0.35:
        text = text[0].lower() + text[1:]
    if not text.endswith(("?", ".")):
        text += random.choice(["?", "?", "?", ".", ""])
    return text

PER_INTENT = 65
records, seen = [], set()
for label, templates in TEMPLATES.items():
    made = 0
    while made < PER_INTENT:
        text = mutate(fill(random.choice(templates)))
        key = text.strip().lower()
        if key in seen:
            continue
        seen.add(key)
        records.append({"text": text, "label": label})
        made += 1

random.shuffle(records)
with open("train_intents.jsonl", "w") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

print(len(records), "training examples")
print(Counter(r["label"] for r in records))
for r in records[:5]:
    print(r)

520 training examples
Counter({'PLAN_QUESTION': 65, 'FEE_QUESTION': 65, 'TAX_ADVICE': 65, 'COMPLAINT': 65, 'ACCOUNT_SUPPORT': 65, 'OTHER': 65, 'TRANSFER_REQUEST': 65, 'COMPARISON_REQUEST': 65})
{'text': 'Am I fully vested yet?', 'label': 'PLAN_QUESTION'}
{'text': 'Hello, what does the plan charge me every year?', 'label': 'FEE_QUESTION'}
{'text': "Hey, what's the penalty for early withdrawal?", 'label': 'TAX_ADVICE'}
{'text': "I'm furious about the fees you charged me", 'label': 'COMPLAINT'}
{'text': "Hello, this company doesn't care about participants?", 'label': 'COMPLAINT'}


In [5]:
import json, re

TEST_ITEMS = [
    ("I need the distribution paperwork to move my retirement savings to an outside IRA.", "TRANSFER_REQUEST"),
    ("Please initiate a direct rollover of my entire vested account to Schwab.", "TRANSFER_REQUEST"),
    ("Now that I've separated from service I want the funds sent to my individual retirement account.", "TRANSFER_REQUEST"),
    ("Sign me up to liquidate the account and wire the proceeds.", "TRANSFER_REQUEST"),
    ("I'd like the whole balance pulled out and parked in an IRA I control.", "TRANSFER_REQUEST"),
    ("Time to move this money - start the transfer to my rollover account.", "TRANSFER_REQUEST"),
    ("Can you cut a check for my account balance payable to my new custodian", "TRANSFER_REQUEST"),
    ("How much does this plan cost me annually in dollars", "FEE_QUESTION"),
    ("I want a breakdown of every charge on my account last year.", "FEE_QUESTION"),
    ("What does the S&P 500 index option charge in basis points", "FEE_QUESTION"),
    ("Do you take a percentage when participants leave", "FEE_QUESTION"),
    ("Show me the total cost of ownership for staying enrolled.", "FEE_QUESTION"),
    ("Why did my statement show a $38 deduction labeled plan administration", "FEE_QUESTION"),
    ("Weigh staying put against moving to an IRA for someone five years from retirement.", "COMPARISON_REQUEST"),
    ("I'm torn between leaving the money with my ex-employer and consolidating - talk me through it.", "COMPARISON_REQUEST"),
    ("Give me the trade-offs of each path before I decide anything.", "COMPARISON_REQUEST"),
    ("Is there any reason not to just merge everything into one IRA", "COMPARISON_REQUEST"),
    ("Break down the advantages of the employer plan versus an individual account.", "COMPARISON_REQUEST"),
    ("Help me think through whether to stay or go.", "COMPARISON_REQUEST"),
    ("What would I lose if I moved the balance out of the plan", "COMPARISON_REQUEST"),
    ("If I convert $20k to Roth this year what does that do to my tax bill", "TAX_ADVICE"),
    ("Walk me through the withholding on an early distribution.", "TAX_ADVICE"),
    ("At what age can I touch the money without the IRS penalty", "TAX_ADVICE"),
    ("Does rolling to a Roth IRA trigger taxes right away", "TAX_ADVICE"),
    ("How is a hardship withdrawal taxed in Connecticut", "TAX_ADVICE"),
    ("What's the most tax-efficient way to start taking money out at 60", "TAX_ADVICE"),
    ("How long do I have to work here before the company match is mine", "PLAN_QUESTION"),
    ("What's the waiting period for new hires to join the 401k", "PLAN_QUESTION"),
    ("Does my employer match dollar for dollar or fifty cents", "PLAN_QUESTION"),
    ("Is there a cliff or does vesting grade over time", "PLAN_QUESTION"),
    ("Can part-time employees participate in the plan", "PLAN_QUESTION"),
    ("What's the Roth versus pre-tax contribution split allowed under the plan", "PLAN_QUESTION"),
    ("Are hardship withdrawals permitted under our plan rules", "PLAN_QUESTION"),
    ("The login page keeps rejecting the password I know is right.", "ACCOUNT_SUPPORT"),
    ("I need to update the email on file.", "ACCOUNT_SUPPORT"),
    ("My quarterly statement never showed up in the mail.", "ACCOUNT_SUPPORT"),
    ("Two-factor texts are going to my old phone number.", "ACCOUNT_SUPPORT"),
    ("The balance on the app doesn't match my paystub contributions.", "ACCOUNT_SUPPORT"),
    ("How do I download my year-to-date transaction history", "ACCOUNT_SUPPORT"),
    ("Three phone calls and nobody has fixed my address change - unbelievable.", "COMPLAINT"),
    ("I was promised a callback last Tuesday and heard nothing.", "COMPLAINT"),
    ("Why was I charged a fee nobody disclosed when I enrolled", "COMPLAINT"),
    ("Every time I call I get a different answer. This is a mess.", "COMPLAINT"),
    ("My distribution sat in processing for six weeks with zero updates.", "COMPLAINT"),
    ("I expect someone to explain why my paperwork was rejected without notice.", "COMPLAINT"),
    ("Do you have a Spanish-speaking line", "OTHER"),
    ("Where are you located", "OTHER"),
    ("Just testing whether this chat saves history.", "OTHER"),
    ("What are your holiday hours", "OTHER"),
    ("Can my spouse call on my behalf", "OTHER"),
]

def tokens(s):
    return set(re.findall(r"[a-z0-9]+", s.lower()))

train = [json.loads(line) for line in open("train_intents.jsonl")]
train_tokens = [tokens(r["text"]) for r in train]

flagged = 0
for text, label in TEST_ITEMS:
    tt = tokens(text)
    for tr in train_tokens:
        overlap = len(tt & tr) / max(1, len(tt | tr))
        if overlap >= 0.7:
            print("POSSIBLE LEAK:", text)
            flagged += 1
            break

with open("test_intents.jsonl", "w") as f:
    for text, label in TEST_ITEMS:
        f.write(json.dumps({"text": text, "label": label}) + "\n")

print(len(TEST_ITEMS), "test examples,", flagged, "flagged for overlap")

50 test examples, 0 flagged for overlap


In [6]:
from google.colab import files
files.download("train_intents.jsonl")
files.download("test_intents.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import json, random, mlflow
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

random.seed(42)

SYSTEM_PROMPT = (
    "You are an intent classifier for a 401(k) retirement plan customer service system. "
    "Classify the customer's message into exactly one intent and respond with JSON only: "
    '{"intent": one of ' + ", ".join(INTENT_LABELS) + ', "confidence": 0.0-1.0, "reason": short phrase}'
)

REASON_BY_LABEL = {
    "TRANSFER_REQUEST": "asks to move money out of the plan",
    "FEE_QUESTION": "asks about fees or costs",
    "COMPARISON_REQUEST": "wants options compared",
    "TAX_ADVICE": "asks about tax consequences",
    "PLAN_QUESTION": "asks about plan rules",
    "ACCOUNT_SUPPORT": "needs account help",
    "COMPLAINT": "expresses dissatisfaction",
    "OTHER": "does not fit a service intent",
}

def to_chat(record):
    target = {
        "intent": record["label"],
        "confidence": round(random.uniform(0.88, 0.99), 2),
        "reason": REASON_BY_LABEL[record["label"]],
    }
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": record["text"]},
        {"role": "assistant", "content": json.dumps(target)},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, enable_thinking=False)}

train_records = [json.loads(line) for line in open("train_intents.jsonl")]
train_ds = Dataset.from_list([to_chat(r) for r in train_records])

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

config = SFTConfig(
    output_dir="lora_intent_adapter",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=20,
    max_length=512,
    fp16=True,
    report_to=[],
)

mlflow.start_run(run_name="lora-intent-v1")
mlflow.log_params({
    "base_model": "Qwen/Qwen3-0.6B", "lora_r": 16, "lora_alpha": 32,
    "lora_dropout": 0.05, "epochs": 2, "lr": 2e-4, "batch": "8x2",
    "train_examples": len(train_records),
})

trainer = SFTTrainer(model=model, args=config, train_dataset=train_ds, peft_config=peft_config)
trainer.train()
trainer.save_model("lora_intent_adapter")
print("adapter saved")

Adding EOS to train dataset:   0%|          | 0/520 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/520 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/520 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/520 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/520 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
20,1.009658
40,0.194101
60,0.147291


adapter saved


In [12]:
import json, re, mlflow
from sklearn.metrics import accuracy_score, f1_score

test_records = [json.loads(line) for line in open("test_intents.jsonl")]
trained_model = trainer.model

def classify(m, text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
    out = m.generate(**inputs, max_new_tokens=80, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def parse_prediction(raw):
    try:
        return json.loads(raw), True
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0)), True
            except json.JSONDecodeError:
                pass
    return {}, False

def evaluate(m, name):
    gold, pred, valid = [], [], 0
    for item in test_records:
        parsed, ok = parse_prediction(classify(m, item["text"]))
        valid += int(ok)
        gold.append(item["label"])
        pred.append(parsed.get("intent", "UNPARSEABLE"))
    acc = accuracy_score(gold, pred)
    f1 = f1_score(gold, pred, average="macro")
    rate = valid / len(test_records)
    print(f"{name}: accuracy={acc:.3f}  macro-F1={f1:.3f}  json-valid={rate:.2%}")
    return {"accuracy": round(acc, 3), "macro_f1": round(f1, 3), "json_validity": round(rate, 3)}

trained_model.disable_adapter_layers()
baseline = evaluate(trained_model, "baseline (no adapter)")
trained_model.enable_adapter_layers()
lora = evaluate(trained_model, "lora")

mlflow.log_metrics({f"baseline_{k}": v for k, v in baseline.items()})
mlflow.log_metrics({f"lora_{k}": v for k, v in lora.items()})
mlflow.end_run()

table = [
    "| model | accuracy | macro-F1 | JSON validity |",
    "|---|---|---|---|",
    f"| Qwen3-0.6B prompted baseline | {baseline['accuracy']} | {baseline['macro_f1']} | {baseline['json_validity']} |",
    f"| Qwen3-0.6B + LoRA adapter | {lora['accuracy']} | {lora['macro_f1']} | {lora['json_validity']} |",
]
with open("results_lora_intent.md", "w") as f:
    f.write("# Intent classifier results: prompted baseline vs LoRA\n\n" + "\n".join(table) + "\n")
print("\n".join(table))

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen3DecoderLayer. Setting `past_key_values=None`.


baseline (no adapter): accuracy=0.000  macro-F1=0.000  json-valid=0.00%
lora: accuracy=0.000  macro-F1=0.000  json-valid=0.00%
| model | accuracy | macro-F1 | JSON validity |
|---|---|---|---|
| Qwen3-0.6B prompted baseline | 0.0 | 0.0 | 0.0 |
| Qwen3-0.6B + LoRA adapter | 0.0 | 0.0 | 0.0 |


In [13]:
for t in ["How do I reset my password", "I want to roll my 401k over to an IRA"]:
    raw = classify(trained_model, t)
    print(repr(raw))

'{"systemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystem'
'{"systemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystem'


In [16]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": "How do I reset my password"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
print(repr(prompt))
print()
print(repr(train_ds[0]["text"]))

'<|im_start|>system\nYou are an intent classifier for a 401(k) retirement plan customer service system. Classify the customer\'s message into exactly one intent and respond with JSON only: {"intent": one of TRANSFER_REQUEST, FEE_QUESTION, COMPARISON_REQUEST, TAX_ADVICE, PLAN_QUESTION, ACCOUNT_SUPPORT, COMPLAINT, OTHER, "confidence": 0.0-1.0, "reason": short phrase}<|im_end|>\n<|im_start|>user\nHow do I reset my password<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

'<|im_start|>system\nYou are an intent classifier for a 401(k) retirement plan customer service system. Classify the customer\'s message into exactly one intent and respond with JSON only: {"intent": one of TRANSFER_REQUEST, FEE_QUESTION, COMPARISON_REQUEST, TAX_ADVICE, PLAN_QUESTION, ACCOUNT_SUPPORT, COMPLAINT, OTHER, "confidence": 0.0-1.0, "reason": short phrase}<|im_end|>\n<|im_start|>user\nAm I fully vested yet?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n{"intent": "PLAN_QUESTION", "confidence

In [18]:
print("=== trained model on a TRAINING item ===")
print(repr(classify(trained_model, train_records[0]["text"])))

print("=== base model, thinking left ON ===")
trained_model.disable_adapter_layers()
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": "How do I reset my password"}]
prompt2 = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt2, return_tensors="pt").to(trained_model.device)
out = trained_model.generate(**inputs, max_new_tokens=250)
print(repr(tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)))
trained_model.enable_adapter_layers()

=== trained model on a TRAINING item ===
'{"systemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystem'
=== base model, thinking left ON ===
'<think>Question Answer* QuestionComparisonarentQuestion Description Answer Instructions InstructionsAddresslicants Answer*Question Question Instructions InstructionsAddress Instructionslicantsarent Answer Answer Answer Answer Navigation Instructions InstructionsComparison InstructionsAddress Solution Questionlicantsarentlicants Instructions Answer Instructionslicants Question Instructionslicantsarentotional Question Instructions Instructi

In [23]:
import json, re, mlflow
from sklearn.metrics import accuracy_score, f1_score

test_records = [json.loads(line) for line in open("test_intents.jsonl")]
trained_model = trainer.model

def classify(m, text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
    out = m.generate(**inputs, max_new_tokens=80)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def parse_prediction(raw):
    try:
        return json.loads(raw), True
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0)), True
            except json.JSONDecodeError:
                pass
    return {}, False

def evaluate(m, name):
    gold, pred, valid = [], [], 0
    for item in test_records:
        parsed, ok = parse_prediction(classify(m, item["text"]))
        valid += int(ok)
        gold.append(item["label"])
        pred.append(parsed.get("intent", "UNPARSEABLE"))
    acc = accuracy_score(gold, pred)
    f1 = f1_score(gold, pred, average="macro")
    rate = valid / len(test_records)
    print(f"{name}: accuracy={acc:.3f}  macro-F1={f1:.3f}  json-valid={rate:.2%}")
    return {"accuracy": round(acc, 3), "macro_f1": round(f1, 3), "json_validity": round(rate, 3)}

trained_model.disable_adapter_layers()
baseline = evaluate(trained_model, "baseline (no adapter)")
trained_model.enable_adapter_layers()
lora = evaluate(trained_model, "lora")

mlflow.log_metrics({f"baseline_{k}": v for k, v in baseline.items()})
mlflow.log_metrics({f"lora_{k}": v for k, v in lora.items()})
mlflow.end_run()

table = [
    "| model | accuracy | macro-F1 | JSON validity |",
    "|---|---|---|---|",
    f"| Qwen3-0.6B prompted baseline | {baseline['accuracy']} | {baseline['macro_f1']} | {baseline['json_validity']} |",
    f"| Qwen3-0.6B + LoRA adapter | {lora['accuracy']} | {lora['macro_f1']} | {lora['json_validity']} |",
]
with open("results_lora_intent.md", "w") as f:
    f.write("# Intent classifier results: prompted baseline vs LoRA\n\n" + "\n".join(table) + "\n")
print("\n".join(table))

baseline (no adapter): accuracy=0.340  macro-F1=0.289  json-valid=100.00%
lora: accuracy=0.800  macro-F1=0.778  json-valid=100.00%
| model | accuracy | macro-F1 | JSON validity |
|---|---|---|---|
| Qwen3-0.6B prompted baseline | 0.34 | 0.289 | 1.0 |
| Qwen3-0.6B + LoRA adapter | 0.8 | 0.778 | 1.0 |


In [24]:
from google.colab import files
files.download("results_lora_intent.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
def gen(m, messages, n=60):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt").to(m.device)
    out = m.generate(**inputs, max_new_tokens=n)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("1. step-2 style, adapter OFF:")
trained_model.disable_adapter_layers()
print(repr(gen(trained_model, [{"role": "user", "content": "In one sentence, what is a 401(k) plan?"}])))
trained_model.enable_adapter_layers()

print("2. step-2 style, adapter ON:")
print(repr(gen(trained_model, [{"role": "user", "content": "In one sentence, what is a 401(k) plan?"}])))

print("3. classifier prompt, test item:")
print(repr(classify(trained_model, "How do I reset my password")))

1. step-2 style, adapter OFF:
'A 401(k) plan is a retirement savings plan offered by employers, allowing employees to save, contribute, and grow money for retirement.'
2. step-2 style, adapter ON:
'A retirement plan offered by employers.'
3. classifier prompt, test item:
'{"intent": "ACCOUNT_SUPPORT", "confidence": 0.9, "reason": "needs account help"}'


In [21]:
model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.generation_config.use_cache = True

print("probe 1, adapter OFF:")
trained_model.disable_adapter_layers()
print(repr(gen(trained_model, [{"role": "user", "content": "In one sentence, what is a 401(k) plan?"}])))
trained_model.enable_adapter_layers()

print("probe 3, classifier prompt:")
print(repr(classify(trained_model, "How do I reset my password")))

probe 1, adapter OFF:
'A 401(k) plan is a retirement savings plan offered by employers to help employees save for retirement.'
probe 3, classifier prompt:
'{"intent": "ACCOUNT_SUPPORT", "confidence": 0.97, "reason": "needs account help"}'


In [9]:
%pip uninstall -y torchao
import mlflow
mlflow.end_run()

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
